In [7]:
import pandas as pd
import os
from google.cloud import bigquery

In [ ]:
# CONFIG
PROJECT_ID = "churn-portfolio-project"
SQL_FILE_PATH = "../sql/feature_engineering_parameterized.sql"
DATA_PATH = "../data/raw/churn_data.csv"

In [30]:
# Initialize BigQuery client
client = bigquery.Client(project=PROJECT_ID)

# Read the SQL query from the file
query = open(SQL_FILE_PATH).read()

# Set up the query job configuration with parameters
job_config = bigquery.QueryJobConfig(
    maximum_bytes_billed=500 * 1024**2, # 500MB safety cap
    query_parameters=[
        bigquery.ScalarQueryParameter("cutoff_date", "DATE", "2025-10-01"),
        bigquery.ScalarQueryParameter("prediction_window_days", "INT64", 30),
    ],
)

/Users/dh/Code/github.com/dsyhub/ecommerce-churn-prediction/.venv/lib/python3.14/site-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [31]:
# Dry Run
job_config.dry_run = True

job = client.query(query, job_config=job_config)

bytes_ = job.total_bytes_processed
mb = bytes_ / 1e6          # MB (decimal)
mib = bytes_ / (1024**2)   # MiB (binary)

print(f"Estimated bytes processed: {bytes_:,}")
print(f"Estimated processed: {mb:.2f} MB  ({mib:.2f} MiB)")

Estimated bytes processed: 9,657,625
Estimated processed: 9.66 MB  (9.21 MiB)


In [32]:
# Real Run
if os.path.exists(DATA_PATH):
    print(f"Loading data from local CSV: {DATA_PATH}")
    df = pd.read_csv(DATA_PATH)
else:
    print("Local data not found. Querying BigQuery...")

    # Run query only when estimated mb < 500
    if mb < 500:
        job_config.dry_run = False
        job = client.query(query, job_config=job_config)
        df = job.to_dataframe()
        df.to_csv(DATA_PATH, index=False)
        print(f"Query executed and saved to local CSV: {DATA_PATH}")

    else:
        print("Too large to run. Please adjust the query parameters.")

Local data not found. Querying BigQuery...


/Users/dh/Code/github.com/dsyhub/ecommerce-churn-prediction/.venv/lib/python3.14/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Query executed and saved to local CSV: ../data/raw/churn_data.csv
